# **Resume / Candidate Screening System**

In [41]:
import re
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

### Data Load

In [42]:
resume = pd.read_csv("/kaggle/input/datasets/snehaanbhawal/resume-dataset/Resume/Resume.csv")
job = pd.read_csv("/kaggle/input/datasets/ravindrasinghrana/job-description-dataset/job_descriptions.csv")

print(f"Resume Text: {resume.shape} \nJob Description: {job.shape}")

Resume Text: (2484, 4) 
Job Description: (1615940, 23)


In [43]:
resume.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [44]:
job.head(3)

,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie..."
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,461-509-4216,Web Developer,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com..."
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,9687619505,Operations Manager,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P..."


In [45]:
resume["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [46]:
resume_text = resume[resume["Category"] == "HR"]
resume_text.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [47]:
job['Job Title'].unique()

array(['Digital Marketing Specialist', 'Web Developer',
       'Operations Manager', 'Network Engineer', 'Event Manager',
       'Software Tester', 'Teacher', 'UX/UI Designer', 'Wedding Planner',
       'QA Analyst', 'Litigation Attorney', 'Mechanical Engineer',
       'Network Administrator', 'Account Manager', 'Brand Manager',
       'Social Worker', 'Social Media Coordinator',
       'Email Marketing Specialist', 'HR Generalist', 'Legal Assistant',
       'Nurse Practitioner', 'Account Director', 'Software Engineer',
       'Purchasing Agent', 'Sales Consultant', 'Civil Engineer',
       'Network Security Specialist', 'UI Developer', 'Financial Planner',
       'Event Planner', 'Psychologist', 'Electrical Designer',
       'Data Analyst', 'Technical Writer', 'Tax Consultant',
       'Account Executive', 'Systems Administrator',
       'Database Administrator', 'Research Analyst', 'Data Entry Clerk',
       'Registered Nurse', 'Investment Analyst', 'Speech Therapist',
       'Sales M

In [48]:
title = job['Job Title'].value_counts()
title["Human Resources Manager"]

np.int64(10180)

In [49]:
job_dataset = job[job["Job Title"] =="Human Resources Manager"]
job_dataset.head(3)

,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
94,955021882863115,1 to 8 Years,MCA,$58K-$83K,Djibouti (city),Djibouti,11.8251,42.5903,Part-Time,38678,...,(374)526-4028,Human Resources Manager,Talent Acquisition Manager,USAJOBS,Talent Acquisition Managers oversee the recrui...,"{'Employee Assistance Programs (EAP), Tuition ...",Talent sourcing Interviewing Onboarding Recrui...,"Lead talent acquisition efforts, including sou...",Titan Company,"{""Sector"":""Consumer Goods"",""Industry"":""Jewelry..."
230,2980724050597001,2 to 12 Years,MCA,$56K-$101K,Pretoria,South Africa,-30.5595,22.9375,Part-Time,17659,...,+1-617-311-3548x974,Human Resources Manager,Talent Acquisition Manager,Glassdoor,Talent Acquisition Managers oversee the recrui...,"{'Employee Referral Programs, Financial Counse...",Talent sourcing Interviewing Onboarding Recrui...,"Lead talent acquisition efforts, including sou...",Glencore,"{""Sector"":""Mining and Metals"",""Industry"":""Mini..."
243,2595804572467199,1 to 9 Years,M.Com,$62K-$102K,BogotÃ¡,Colombia,4.5709,-74.2973,Full-Time,27095,...,894.991.5127,Human Resources Manager,Employee Relations Specialist,Jobs2Careers,Employee Relations Specialists focus on mainta...,"{'Tuition Reimbursement, Stock Options or Equi...",Employee conflict resolution HR compliance Emp...,"Address employee concerns, resolve conflicts, ...",Huawei Technologies,"{""Sector"":""Technology and Networking"",""Industr..."


### Get useful data columns

In [50]:
resume_text = pd.DataFrame(resume_text['Resume_str']).reset_index(drop=True)
resume_text.head(3)

,Resume_str
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,"HR SPECIALIST, US HR OPERATIONS ..."
2,HR DIRECTOR Summary Over 2...


In [51]:
job_dataset = job_dataset[["Job Description"]].drop_duplicates().reset_index(drop=True)
job_dataset.head(3)

,Job Description
0,Talent Acquisition Managers oversee the recrui...
1,Employee Relations Specialists focus on mainta...
2,"HR Generalists handle various HR functions, in..."


## **Resume Data Preprocessing**

### Resume to Skills extract

In [52]:
# Here, only those resume strings have been selected in which skills are mentioned once or twice.
s = []
s_1 = []
s_2 = []

for index, i in enumerate(range(len(resume_text))):
    con = resume_text["Resume_str"][i].count("Skills")
    s.append(con)
    if con == 1:
        s_1.append(index)
    elif con == 2:
        s_2.append(index)
    
print(f"Total Resume: {len(resume_text)}")
print(f"Resume in One time Skills : {s.count(1)}")
print(f"          Two time Skills : {s.count(2)}")

Total Resume: 110
Resume in One time Skills : 73
          Two time Skills : 23


In [53]:
resume_text = resume_text.loc[s_1]
resume_text.head(2)

,Resume_str
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,"HR SPECIALIST, US HR OPERATIONS ..."


In [54]:
def Resume_Skills_extract(resume_text):
    Resume_skills = []

    # Split Resume Text skills keyword wise (1 times Split)
    for i in s_1:
        Resume_skills.append(resume_text.split("Skills")[1]) 

    # comma based split
    sp_skill = []
    for i in range(len(Resume_skills)):
        sp_skill.append(Resume_skills[i].split(","))


    # Filtering Skill meaning max 50 and remove big string
    n_sp_skill = []
    for i in range(len(sp_skill)):
        row = []
        for j in range(len(sp_skill[i])):
            if len(sp_skill[i][j]) < 50:
                row.append(sp_skill[i][j].lower())
        n_sp_skill.append(row)

    return n_sp_skill[0]


In [55]:
# Remove None rows
resume_text = resume_text.dropna(axis=0)
resume_text = pd.DataFrame(resume_text['Resume_str'], columns=['Resume_str'])

#### **Resume Dataset Cleaning And Preprocessing**

In [56]:
def Resume_Preprocessing(text):
    text = re.sub("\n", " ", text)
    text = re.sub(r"\s+", " ", text).lower()
    return text

#### **Job Dataset Preprocessing**

In [57]:
job_skills = [
            "hr", "policies", "benefits", "payroll", "recruitment", "human resources", "personnel", "recruiting",
            "hiring", "employee relations", "excel", "processes", "staffing", "office", "performance management", "leadership", 
            "hris", "quality", "insurance", "organizational"
] 

In [58]:
job_dataset

,Job Description
0,Talent Acquisition Managers oversee the recrui...
1,Employee Relations Specialists focus on mainta...
2,"HR Generalists handle various HR functions, in..."


### Load Embedding Model (Sentence Transformer)

In [59]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# **Finally Similarty Evaluate**

In [78]:
def job_and_resume_skill_match(resume_skills, job_skills):
    match = []
    not_match = []

    # resume skills ko clean karke set me convert kar diya
    resume_set = set(skill.strip().lower() for skill in resume_skills)

    for skill in job_skills:
        cleaned_skill = skill.strip().lower()

        if cleaned_skill in resume_set:
            match.append(skill)
        else:
            not_match.append(skill)

    return {
        "Matched Skills": match,
        "Not Matched Skills": not_match[:7]
    }

In [79]:
def get_similarty_and_match_skill(resume_text, job_des, job_skills):
    resume_emb = embed_model.encode(resume_text)
    job_emb = embed_model.encode(job_des)
    sim = cosine_similarity([resume_emb], [job_emb])

    ext_resume_skill = Resume_Skills_extract(resume_text)
    skill_matching = job_and_resume_skill_match(ext_resume_skill, job_skills)

    return {
        "Job to Resume Similarity" : f"{int(sim[0][0]*100)}%",
        "Matchs:" : skill_matching
        }  

In [80]:
def Display_Modify(obj):
    print(f"Job to Resume Similarty: {obj['Job to Resume Similarity']}")
    print(f"Match Skills: {obj['Matchs:']['Matched Skills']}")
    print(f"NOT Match Skills: {obj['Matchs:']['Not Matched Skills']}")

### **Test 1**

In [81]:
obj = get_similarty_and_match_skill(resume_text["Resume_str"][1],job_dataset["Job Description"][0], job_skills)
Display_Modify(obj)

Job to Resume Similarty: 29%
Match Skills: ['hr', 'recruitment', 'quality']
NOT Match Skills: ['policies', 'benefits', 'payroll', 'human resources', 'personnel', 'recruiting', 'hiring']


### **Test 2**

In [83]:
obj2 = get_similarty_and_match_skill(resume_text["Resume_str"][0],job_dataset["Job Description"][2], job_skills)
Display_Modify(obj2)

Job to Resume Similarty: 46%
Match Skills: ['policies', 'benefits', 'payroll', 'human resources', 'personnel', 'employee relations', 'office', 'insurance', 'organizational']
NOT Match Skills: ['hr', 'recruitment', 'recruiting', 'hiring', 'excel', 'processes', 'staffing']
